## Root Cause Analysis - Exp2

* Goals
  * Root Cause Analysis on individual RAG results --> 2 RAGs results comparison
  * Provide actionable suggestions
* About Exp2
  * Build Langgraph Agent System for Root Cause Analysis

In [1]:
%load_ext autoreload
%autoreload 2

import os
from sqlalchemy import make_url
import pandas as pd

from utils_root_cause import *

from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Optional
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
import json

import warnings
warnings.filterwarnings('ignore')


exp_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.78
)

DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
DATABASE_URL = DATABASE_URL_PUBLIC.replace("postgres://", "postgresql://")
db_url = make_url(DATABASE_URL)

In [2]:
auto_eval_output = get_auto_eval_output(db_url).iloc[:5]
print(auto_eval_output.shape)

auto_eval_output.head(n=2)

(5, 16)


,config_hash,dataset,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,query,context,retrieved_content,same_context,retrieval_quality_score,rq_reasoning,referenced_answer,ai_answer,answer_quality_score,aq_reasoning
0,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,true,3,The retrieved content reproduces the entire co...,Have the check reissued to the proper payee.Ju...,To deposit a cheque issued to an associate in ...,2,The AI’s answer correctly identifies that the ...
1,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,true,3,The retrieved content matches the context exac...,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...",2,The AI’s answer correctly states that a busine...


In [3]:
# =========================
# State Schema
# =========================
class RCAState(TypedDict):
    r_idx: int
    query: str
    referenced_content: str
    retrieved_content: str
    retrieval_quality_score: int
    rq_reasoning: str

    embedding_model: str
    top_n_retrieval: int
    semantic_weight: float
    answer_gen_llm: str

    referenced_answer: str
    ai_answer: str
    answer_quality_score: int
    aq_reasoning: str

    new_retrieval_quality_score: int   
    new_answer_quality_score: int
    query_quality: str     
    root_cause_analysis: list[dict[str, str]]  # dict key is root cause, value is improvement suggestions
    re_eval_lst: set[int]  # store the index of records that should be re-evaluated after updates

    referenced_answer_context_alignment: Optional[int]  # indicate whether referenced answer has critical info alignede with the referenced content
    referenced_answer_context_alignment_reason: Optional[str]
    ai_answer_context_alignment: Optional[int]  # indicate whether AI answer has critical info alignede with the referenced content
    ai_answer_context_alignment_reason: Optional[str]

In [4]:
# =========================
# AI-as-Judge Reviewer
# =========================
async def review_evaluation_output(state: RCAState):
    rq_score_after_review = await review_sr_async(exp_llm, state['rq_reasoning'])
    aq_score_after_review = await review_sr_async(exp_llm, state['aq_reasoning'])

    root_cause_analysis = []
    if rq_score_after_review != state['retrieval_quality_score']:
        root_cause_analysis.append({'auto eval score issue': f'score vs reasoning mismatch, \
                                    changed retrieval_quality_score from {state["retrieval_quality_score"]} to {rq_score_after_review}'})
    if aq_score_after_review != state['answer_quality_score']:
        root_cause_analysis.append({'auto eval score issue': f'score vs reasoning mismatch, \
                                    changed answer_quality_score from {state["answer_quality_score"]} to {aq_score_after_review}'})

    return {
        "new_retrieval_quality_score": rq_score_after_review,
        "new_answer_quality_score": aq_score_after_review,
        "root_cause_analysis": root_cause_analysis
    }


# =========================
# Reference Reviewer
# =========================
async def review_referenced_data(state: RCAState):
    rq_score = state['new_retrieval_quality_score']
    aq_score = state['new_answer_quality_score']
    root_cause_analysis = state['root_cause_analysis']
    re_eval_lst = set()

    rc_alignment_score, rc_alignment_reason = None, None
    ac_alignment_score, ac_alignment_reason = None, None

    if rq_score == -1:
        root_cause_analysis.append({'Please review referenced content':
                                            'referenced content is much less relevant to the query than retrieved content'})
        re_eval_lst.add(state['r_idx'])
    if aq_score == -1:
        root_cause_analysis.append({"Please review referenced answer":
                                            "referenced answer is much less relevant to the query than AI's answer"})
        re_eval_lst.add(state['r_idx'])

    if rq_score >= 2 and aq_score in [0, 1]:  # good retrieval, bad answer
        referenced_answer_context_alignment = await review_ta_async(exp_llm,state['query'],
                                                                    state['referenced_answer'], state['referenced_content'])
        rc_alignment_score = referenced_answer_context_alignment.score
        rc_alignment_reason = referenced_answer_context_alignment.reasoning
        if rc_alignment_score == 0:
            root_cause_analysis.append({'Please review referenced answer':
                                        'referenced answer has critical information misaligned with the referenced content'})
            re_eval_lst.add(state['r_idx'])
            
    if rq_score in [0, 1] and aq_score >= 2:  # good answer, bad retrieval
        ai_answer_context_alignment = await review_ta_async(exp_llm,state['query'],
                                                                    state['ai_answer'], state['referenced_content'])
        ac_alignment_score = ai_answer_context_alignment.score
        ac_alignment_reason = ai_answer_context_alignment.reasoning
        if ac_alignment_score == 0:
            root_cause_analysis.append({"Please review referenced content':\
                                        AI's answer got a high score but retrieval score is low, \
                                        please check whether need to add or merge retrieved content into the referenced content."})
            re_eval_lst.add(state['r_idx'])

    return {
        "referenced_answer_context_alignment": rc_alignment_score,
        "referenced_answer_context_alignment_reason": rc_alignment_reason,
        "ai_answer_context_alignment": ac_alignment_score,
        "ai_answer_context_alignment_reason": ac_alignment_reason,
        "root_cause_analysis": root_cause_analysis,
        "re_eval_lst": re_eval_lst
    }


# =========================
# Query Quality Reviewer
# =========================
async def review_query_quality(state: RCAState):
    query_quality = await review_query_quality_async(exp_llm, state['query'])
    root_cause_analysis = state['root_cause_analysis']

    if query_quality == 'ambiguous':
        query_variants = await expand_query_async(exp_llm, state['query'])
        root_cause_analysis.append({'Please review query quality': query_variants})

    return {
        "query_quality": query_quality,
        "root_cause_analysis": root_cause_analysis
    }

In [5]:
workflow = StateGraph(RCAState)

workflow.add_node("Evaluation Reviewer", review_evaluation_output)
workflow.add_node("Reference Reviewer", review_referenced_data)
workflow.add_node("Query Quality Reviewer", review_query_quality)

workflow.add_edge(START, "Evaluation Reviewer")
workflow.add_edge("Evaluation Reviewer", "Reference Reviewer")
workflow.add_edge("Reference Reviewer", "Query Quality Reviewer")
workflow.add_edge("Query Quality Reviewer", END)

graph = workflow.compile()

In [6]:
# # =========================
# # RAG System Reviewer
# # query quality, system prompt, configs
# # =========================
# async def review_rag_system(state: RCAState):
#     system_prompt = FINANCIAL_RAG_SYSTEM_PROMPT
#     user_query = state['query']
#     query_quality = await review_query_quality_async(exp_llm, user_query)
#     rag_config = f"""
#                  - Embedding model: {state['embedding_model']}
#                  - Top N retrieval: {state['top_n_retrieval']}
#                    * It specifies the number of top relevant documents retrieved to assist answer generation.
#                  - Semantic weight: {state['semantic_weight']}
#                    * Semantic weight means the percentage of semantic retrieval, 1 - semantic_weight is the percentage of keyword retrieval.
#                  - Answer generation LLM: {state['answer_gen_llm']}
#                    * The LLM used to generate the answer based on the retrieved content and the query.
#                  """
#     retrieval_quality_score = state['new_retrieval_quality_score']
#     answer_quality_score = state['new_answer_quality_score']
#     root_cause_analysis = state['root_cause_analysis']

#     # no issue, or needs to re-evaluate, pass RAG system review
#     if (retrieval_quality_score == 3 and answer_quality_score == 3) or len(state['re_eval_lst']) > 0:
#         return {"root_cause_analysis": root_cause_analysis}
    
#     rq_reasoning = state['rq_reasoning']
#     aq_reasoning = state['aq_reasoning']
    
#     rag_sys_review = await review_rag_system_async(exp_llm, retrieval_quality_score, rq_reasoning,
#                                    answer_quality_score, aq_reasoning,
#                                    user_query, query_quality, system_prompt, rag_config)
    
#     root_cause_analysis.append({f'RAG Sys Review: {rag_sys_review.root_cause_analysis}': rag_sys_review.improvement_suggestions})
    
#     return {"root_cause_analysis": root_cause_analysis}

In [7]:
async def run_rca_on_all(df, max_concurrent=2):
    sem = asyncio.Semaphore(max_concurrent)
    async def throttled_invoke(state):
        async with sem:
            return await graph.ainvoke(state)

    states = [
        {
            "r_idx": idx,
            "query": row["query"],
            "referenced_content": row["context"],
            "retrieved_content": row["retrieved_content"],
            "retrieval_quality_score": int(row["retrieval_quality_score"]),
            "rq_reasoning": row["rq_reasoning"],
            "referenced_answer": row["referenced_answer"],
            "ai_answer": row["ai_answer"],
            "answer_quality_score": int(row["answer_quality_score"]),
            "aq_reasoning": row["aq_reasoning"],
            "embedding_model": row["embedding_model"],
            "top_n_retrieval": row["top_n_retrieval"],
            "semantic_weight": row["semantic_weight"],
            "answer_gen_llm": row["answer_gen_llm"]
        }
        for idx, row in df.iterrows()
    ]
    return await asyncio.gather(*[throttled_invoke(s) for s in states])

rca_results = await run_rca_on_all(auto_eval_output)

In [8]:
rca_results_df = pd.DataFrame(rca_results)
display(rca_results_df.head())

diff_df = rca_results_df[(rca_results_df["retrieval_quality_score"] != rca_results_df["new_retrieval_quality_score"]) \
                         | (rca_results_df["answer_quality_score"] != rca_results_df["new_answer_quality_score"])]
display(diff_df.head())

,r_idx,query,referenced_content,retrieved_content,retrieval_quality_score,rq_reasoning,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,...,aq_reasoning,new_retrieval_quality_score,new_answer_quality_score,query_quality,root_cause_analysis,re_eval_lst,referenced_answer_context_alignment,referenced_answer_context_alignment_reason,ai_answer_context_alignment,ai_answer_context_alignment_reason
0,0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content reproduces the entire co...,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,...,The AI’s answer correctly identifies that the ...,3,2,clear,[],{},None,None,None,None
1,1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content matches the context exac...,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,...,The AI’s answer correctly states that a busine...,3,2,clear,[],{},None,None,None,None
2,2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,1,The retrieved content discusses the need for a...,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,...,The AI's answer directly addresses the user’s ...,2,-1,ambiguous,[{'auto eval score issue': 'score vs reasoning...,{2},None,None,None,None
3,3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...,3,The retrieved content exactly matches the cont...,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,...,The AI’s answer is relevant to the query and c...,3,2,clear,[],{},None,None,None,None
4,4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...,3,The retrieved content is identical to the cont...,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,...,The AI’s answer addresses the difficulty of ac...,3,2,clear,[],{},None,None,None,None


,r_idx,query,referenced_content,retrieved_content,retrieval_quality_score,rq_reasoning,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,...,aq_reasoning,new_retrieval_quality_score,new_answer_quality_score,query_quality,root_cause_analysis,re_eval_lst,referenced_answer_context_alignment,referenced_answer_context_alignment_reason,ai_answer_context_alignment,ai_answer_context_alignment_reason
2,2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,1,The retrieved content discusses the need for a...,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,...,The AI's answer directly addresses the user’s ...,2,-1,ambiguous,[{'auto eval score issue': 'score vs reasoning...,{2},None,None,None,None


In [18]:
for lst in diff_df["root_cause_analysis"].values:
    for elem in lst:
        print(json.dumps(elem, indent=4, ensure_ascii=False))

{
    "auto eval score issue": "score vs reasoning mismatch,                                     changed retrieval_quality_score from 1 to 2"
}
{
    "Please review referenced answer": "referenced answer is much less relevant to the query than AI's answer"
}
{
    "Please review query quality": [
        "Can a single EIN be associated with multiple doing‑business‑as names, and how can I find those names?",
        "What is the process for registering multiple DBA names under one EIN?",
        "How do I list all DBA names linked to a specific EIN?"
    ]
}


In [9]:
rca_results_df.to_csv("rca_results.csv", index=False)

In [13]:
import pandas as pd
import json
import ast


rca_results_df = pd.read_csv("rca_results.csv")
data = ast.literal_eval(rca_results_df.iloc[0]["root_cause_analysis"])[0]

for key, value in data.items():
    print("\n=== REVIEW ===\n")
    print(key)

    print("\n=== SUGGESTIONS ===\n")
    print(value)


=== REVIEW ===

RAG Sys Review: The answer quality score is limited primarily by the narrow retrieval and strict prompt constraints. With only one document retrieved (Top N = 1) and a 50/50 semantic/keyword split, the system often receives an incomplete set of facts even though the single document may contain most of the needed information. The system prompt explicitly forbids the LLM from using any knowledge outside the retrieved content and discourages speculation, which leads the model to omit potentially relevant points that it cannot confirm from the single snippet. Additionally, the LLM may not fully surface all details present in the snippet due to token limits or conservative generation behavior, resulting in a partially complete answer despite the retrieval score being high. These factors together prevent the answer quality from reaching the maximum score of 3.

=== SUGGESTIONS ===

1. **Increase Top N retrieval**: fetch 3–5 documents to provide a broader context and cover al